# 40장 · Streamlit 폼으로 만드는 RAG 질의 화면

Streamlit의 폼은 입력을 모았다가 제출 시점에 처리합니다. 작성 중인 질문과 마지막으로 제출한 질문을 구분하면 화면의 답변이 언제 바뀌어야 하는지 알 수 있습니다.

Streamlit 폼을 제출할 때만 검색하고, 새 응답의 답변과 출처를 함께 저장합니다. 대출 질문 뒤에 근거가 없는 주차 질문을 보내 이전 출처가 패널에 남는지 검사합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 제출 이벤트와 검색 결과 갱신

Streamlit은 위젯을 조작하면 화면 코드를 다시 실행합니다. form으로 묶은 입력은 제출 버튼을 누르는 시점에 함께 받습니다. 새 실행을 지나도 보여 줄 값은 session_state에 남깁니다. 먼저 노트북에서 상태에 어떤 값이 들어가야 하는지 확인합니다.


### 셀 01 · submit()으로 현재 결과 갱신

화면에 대출 답변의 출처가 표시돼 있습니다. 주차 질문을 보냈을 때 이전 출처가 남는지 살펴보세요.


**실행 준비** · 30장의 문서 세 건과 retrieve를 불러옵니다. 아래 준비 셀을 한 번 실행한 뒤 이어지는 예제에서 이번 장의 변경을 확인하세요.

In [ ]:
records = [{'id': 'library-1',
  'text': '도서 대출 기간은 14일입니다.',
  'source': '가상 도서관 이용 안내',
  'page': 1},
 {'id': 'library-2',
  'text': '도서관은 월요일에 휴관합니다.',
  'source': '가상 도서관 이용 안내',
  'page': 2},
 {'id': 'library-3',
  'text': '무인 반납함은 24시간 이용할 수 있습니다.',
  'source': '가상 도서관 이용 안내',
  'page': 3}]
terms = ("대출", "휴관", "반납")

def retrieve(question, k=1):
    query = [int(term in question) for term in terms]
    if not any(query):
        return []
    def distance(row):
        vector = [int(term in row["text"]) for term in terms]
        return sum((a-b)**2 for a, b in zip(query, vector)), row["id"]
    return sorted(records, key=distance)[:k]


In [1]:
view = {'question': '대출 기간은?', 'sources': [records[0]]}
def submit(question):
    found = retrieve(question, 1)
    return {'question': question, 'sources': found}

view = submit('주차 요금은?')
print(view)

{'question': '주차 요금은?', 'sources': []}


sources가 빈 목록으로 바뀌었습니다. 기존 화면에 새 답만 덧씌우면 예전 출처가 남을 수 있습니다. 질문과 결과를 한 번에 갱신한 까닭입니다.

처음에 근거가 있는 질문을 하고 바로 이어 주차 요금을 물어보세요. 두 번째 답변의 출처 칸에는 첫 질문의 문서가 남아 있으면 안 됩니다. 빈 sources도 화면에 반영해야 직전에 읽은 원문이 이번 답의 근거처럼 보이지 않습니다.


## Streamlit 폼과 AppTest 입력

다음 문자열은 실제 Streamlit 앱 코드입니다. 실습 범위를 입력·출력 위젯으로 좁히려고 대출과 휴관 원문 두 건만 씁니다. 수업 커널에 Streamlit이 있으면 AppTest가 앱을 메모리에서 실행해 위젯을 조작합니다. AppTest는 브라우저 화면을 그리지 않습니다.


### 셀 02 · Streamlit 폼 구성

if submitted 안에서 찾은 문서가 last에 저장됩니다. 그 아래에서는 저장된 문서가 있는지에 따라 안내 문구와 출처를 나누어 표시합니다.


In [2]:
app_source = """import streamlit as st

data = {"대출": ("도서 대출 기간은 14일입니다.", "library-1", 1),
        "휴관": ("도서관은 월요일에 휴관합니다.", "library-2", 2)}

st.title("가상 도서관에 질문하기")
with st.form("question_form"):
    question = st.text_input("질문")
    submitted = st.form_submit_button("질문 보내기")
if submitted:
    hit = next((row for word, row in data.items() if word in question), None)
    st.session_state["last"] = hit
    st.session_state["asked"] = True
if st.session_state.get("asked"):
    hit = st.session_state.get("last")
    if hit is None:
        st.info("제공된 자료에서 확인할 수 없습니다.")
    else:
        st.write(hit[0])
        st.caption(f"근거 [{hit[1]}] 가상 도서관 이용 안내 / {hit[2]}쪽")
"""
import ast
app_tree = ast.parse(app_source)
print(app_tree.body[0].names[0].name)

streamlit


streamlit은 소스의 첫 import에서 읽은 이름입니다. 앱의 구문이 맞아도 버튼을 눌렀을 때의 동작까지 확인한 것은 아닙니다. 다음 셀에서 같은 app_source 문자열을 AppTest에 넘겨 폼 제출을 재현합니다.


#### 화면이 다시 그려질 때의 검색

<mark>폼 안에서 내용을 편집한 일과 제출 버튼을 누른 일은 서로 다른 사건입니다.</mark> 주차 요금을 입력했을 때 검색 결과가 비었다면 이전 질문의 출처도 함께 비워야 합니다. 현재 입력과 마지막 제출 결과를 구분해 읽으세요.

선수 개념 · 30장 LangGraph의 검색 노드와 답변 노드

#### 제출할 때 last를 덮어쓰는 Streamlit 폼

```mermaid
flowchart TD
nform["질문 입력"]
  nsubmit["폼 제출"]
  nlast["상태 저장"]
  nhit["답변·쪽수"]
  nmiss["자료 없음"]
  nform --> nsubmit
  nsubmit --> nlast
  nlast --> nhit
  nlast --> nmiss
```

앱 코드의 휴관 질문을 따라갑니다. asked는 제출 여부를 기억하고 last의 내용이 결과 표시를 결정합니다.

### 제출 전과 제출 후의 화면

<p>대출 답이 보이는 상태에서 주차 질문을 제출해 보세요. 검색 결과가 비었을 때도 last를 갱신해야 다음 화면에서 대출 답과 출처가 함께 사라집니다.</p>


### asked와 last로 현재 화면 결정하기

<p>asked는 폼을 제출한 적이 있는지, last는 방금 찾은 문서가 무엇인지 담습니다. 첫 화면과 답을 찾지 못한 화면은 last가 비어 있을 수 있지만 asked 값으로 구별됩니다.</p>


### 셀 03 · AppTest로 폼 제출

Streamlit이 설치된 수업 커널에서는 첫 입력칸과 제출 버튼을 실제로 조작합니다. 설치 여부에 따라 출력이 달라집니다.

같은 app 객체에서 대출 질문을 제출한 뒤 주차 질문을 제출합니다. 마지막 검사에서 첫 질문의 캡션이 남아 있으면 코드가 멈춥니다.


In [3]:
from importlib.util import find_spec
if find_spec('streamlit') is None:
    print('Streamlit이 설치된 수업 커널에서 폼 실습을 실행하세요.')
else:
    from streamlit.testing.v1 import AppTest
    app = AppTest.from_string(app_source).run()
    app.text_input[0].set_value('대출 기간은?')
    app.button[0].click().run()
    assert not app.exception
    print(app.markdown[0].value, app.caption[0].value)
    app.text_input[0].set_value('주차 요금은?')
    app.button[0].click().run()
    assert not app.exception and not app.caption
    print(app.info[0].value)

Streamlit이 설치된 수업 커널에서 폼 실습을 실행하세요.


Streamlit이 설치된 커널에서는 대출 답과 1쪽 캡션을 읽은 뒤 주차 질문을 제출합니다. 두 번째 응답에서 캡션이 지워지고 자료 없음 안내가 표시되어야 합니다. 설치 안내만 출력된 커널에서는 위젯 동작을 확인하지 못한 상태입니다.


### 과제 04 · 검색 결과의 표시 문구

submit으로 휴관 자료를 찾아 제목과 쪽수가 보이는 출처 문구를 작성하세요.


In [4]:
# 휴관 자료 하나에서 source와 page를 꺼내세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>current = submit(&#x27;휴관일은?&#x27;)
source = current[&#x27;sources&#x27;][0]
print(f&quot;{source[&#x27;source&#x27;]} / {source[&#x27;page&#x27;]}쪽&quot;)</code></pre>
<p>실행 결과</p><pre><code>가상 도서관 이용 안내 / 2쪽</code></pre>
<p>2쪽이라는 위치가 나옵니다. 화면에서 링크나 문구를 만들 때 목록의 순번을 쪽수로 쓰면 원문을 찾아갈 수 없습니다.</p>
</details>


### 쪽 번호가 있는 출처 문구

<p>과제에서 받은 휴관 문서는 결과 목록의 첫 번째 항목이지만 원문에서는 2쪽입니다. 화면에는 검색 순위 대신 문서에 저장된 page를 사용합니다.</p>


### 과제 05 · 제출 결과로 출처 패널 교체

submit으로 대출 질문을 제출한 뒤 같은 view에 주차 질문을 제출하세요. 두 차례 모두 현재 sources에서 표시할 쪽 번호를 구하세요. 새 응답의 빈 목록이 화면에 반영되는지 확인하세요.


In [ ]:
# 대출 결과를 view에 넣고 page 목록을 출력하세요. 이어 주차 질문으로 교체하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>view = submit(&#x27;대출 기간은?&#x27;)
print([doc[&#x27;page&#x27;] for doc in view[&#x27;sources&#x27;]])
view = submit(&#x27;주차 요금은?&#x27;)
print([doc[&#x27;page&#x27;] for doc in view[&#x27;sources&#x27;]])</code></pre>
<p>실행 결과</p><pre><code>[1]
[]</code></pre>
<p>두 번째 제출에서 1쪽이 사라집니다. 이 셀은 노트북의 상태 갱신을 검사합니다. Streamlit이 설치된 커널에서는 셀 03의 app으로 이 두 질문을 순서대로 제출해 실제 위젯의 캡션도 비워졌는지 확인할 수 있습니다.</p>
</details>


## 돌아보기

정상 답 뒤에 자료 없는 질문을 보내 출처가 지워지는지 확인하세요. 웹 수업 화면까지 점검할 때는 폼 제출 전후와 화면을 다시 그린 뒤를 나누어 보세요.


## 폼을 다시 그려도 지난 답이 중복되지 않는 대화 기록 만들기

앞에서 만든 폼은 마지막 검색 결과를 화면에 표시합니다. 사용자에게 방금 전 질문도 보여 주려면 화면이 다시 그려지는 횟수와 제출 횟수를 구분해서 대화 기록을 쌓아야 합니다.

### 셀 07 · 한 번 제출했을 때만 추가하는 함수

새 항목을 만드는 일과 기존 항목을 그리는 일을 나눕니다. 제출 여부와 질문을 받아 새 기록을 반환하는 함수를 작성한 예입니다. 기존 submit이 실제 검색을 수행합니다.

In [5]:
def after_form(history, submitted, question):
    if not submitted or not question.strip():
        return history
    return [*history, submit(question.strip())]

conversation = after_form([], True, "반납함 이용 시간은?")
redrawn = after_form(conversation, False, "반납함 이용 시간은?")
print(len(redrawn), redrawn[0]["sources"][0]["id"])

1 library-3


제출 한 번과 화면 갱신 한 번 뒤에도 남는 질문은 한 개입니다. 여기서는 다시 그린 횟수를 따로 세지 않고 submitted가 참일 때만 새 행을 만듭니다.

### 과제 08 · 빈 결과인 두 번째 질문도 기록에 남기기

주차 질문을 제출한 뒤 두 항목을 순서대로 읽으세요. 각 항목에서 자체 sources만 이용해 질문과 근거 ID를 출력합니다. 근거가 없으면 ‘자료 없음’을 표시하세요.

In [6]:
# after_form으로 주차 질문을 제출하고 각 행의 sources를 확인하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>log_rows = after_form(redrawn, True, &quot;주차 요금은?&quot;)
for turn in log_rows:
    evidence = turn[&quot;sources&quot;]
    label = evidence[0][&quot;id&quot;] if evidence else &quot;자료 없음&quot;
    print(turn[&quot;question&quot;], label)</code></pre>
<p>실행 결과</p><pre><code>반납함 이용 시간은? library-3
주차 요금은? 자료 없음</code></pre>
<p>첫 줄은 library-3, 둘째는 자료 없음입니다. 근거가 마지막으로 성공한 질문의 전역 변수에만 저장돼 있다면 주차 질문에도 반납함 출처를 붙이는 오류가 생깁니다.</p>
</details>

### 셀 09 · 선택 실습 · 실제 폼 두 번 제출하고 화면 기록 읽기

Streamlit을 설치한 커널에서 AppTest로 폼을 실제로 누릅니다. 화면은 질문과 최근 자료 유무를 두 줄에 남깁니다. 설치 전에도 바로 앞 과제의 검색·기록 로직을 검산할 수 있습니다.

In [7]:
from importlib.util import find_spec
from textwrap import dedent
history_app_source = dedent("""
    import streamlit as st
    notices = {"반납": "library-3", "대출": "library-1"}
    with st.form("ask"):
        question = st.text_input("질문")
        sent = st.form_submit_button("보내기")
    log = st.session_state.setdefault("log", [])
    if sent and question.strip():
        hit = next((value for word, value in notices.items() if word in question), "자료 없음")
        log.append((question, hit))
    for asked, evidence in log:
        st.text(f"{asked} · {evidence}")
""")
if find_spec("streamlit") is not None:
    from streamlit.testing.v1 import AppTest
    learner_screen = AppTest.from_string(history_app_source).run()
    for question in ("반납함 이용 시간은?", "주차 요금은?"):
        learner_screen.text_input[0].set_value(question)
        learner_screen.button[0].click().run()
    learner_screen.run()
    assert not learner_screen.exception
    print([line.value for line in learner_screen.text])
else:
    print("Streamlit 미설치 · 화면 재실행은 건너뜁니다.")

Streamlit 미설치 · 화면 재실행은 건너뜁니다.


실제 폼에서 마지막 run()은 버튼을 다시 누르지 않고 화면만 갱신합니다. 두 질문이 한 줄씩 보이면 그려 주는 코드가 세 번째 질문을 만들지 않은 것입니다. 설치 안내만 나왔다면 이 셀의 화면 동작은 아직 확인하지 않았습니다.

<mark>질문을 저장하는 코드는 제출 시점에, 쌓인 기록을 그리는 코드는 폼 바깥에 둡니다.</mark> 폼을 그릴 때마다 append를 실행하면 새 질문을 보내지 않아도 마지막 줄이 늘어납니다.

41장에서 화면의 대화 ID와 그래프에 보낼 thread_id를 같이 다룹니다.